In [4]:
import time
import math
import matplotlib.pyplot as plt
from dds import DDS

In [5]:
class PID_Controller:
    def __init__(self, kp, ki, kd):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.integral = 0.0
        self.prev_error = 0.0

    # Aggiungiamo max_output come parametro opzionale
    def evaluate(self, error, dt, max_output=None):
        p = self.kp * error
        
        if dt > 0.0:
            d = self.kd * (error - self.prev_error) / dt
        else:
            d = 0.0

        if max_output is not None:
            potential_output = p + (self.ki * self.integral) + d
            
            is_saturated = abs(potential_output) >= max_output
            
            same_direction = (potential_output > 0 and error > 0) or (potential_output < 0 and error < 0)
            
            if is_saturated and same_direction:
                pass 
            else:
                self.integral += error * dt
        else:
            self.integral += error * dt

        i = self.ki * self.integral
        self.prev_error = error

        final_output = p + i + d

        if max_output is not None:
            final_output = max(-max_output, min(max_output, final_output))

        return final_output

In [9]:
dds = DDS()
dds.start('127.0.0.1', 4444)
dds.subscribe(['tick', 'X', 'Z', 'Yaw', 'Speed', 'X_dest', 'Z_dest', 'Robot_index', 'Trajectory_Type'])

acceptance_radius = 2
slow_radius = 5

max_speed = 3
target_speed = 3
max_torque = 150

speed_pid = PID_Controller(20.0, 100.0, 0.05)
steering_pid = PID_Controller(kp=1.0, ki=0.0, kd=0.0)

max_steering = 35

d_x = []
d_z = []   
d_time = []
d_target_speed = []
d_current_speed = []
d_angle_error = []
d_steering = []

print("Starting robot conrtrol...")
start_time = time.time()

try:
    while True:
        delta_t = dds.wait('tick')
        if delta_t is None:
            continue
            
        t_elapsed = time.time() - start_time

        x = dds.read('X')
        z = dds.read('Z')
        yaw = dds.read('Yaw') 
        current_speed = dds.read('Speed')
        target_x = dds.read('X_dest')
        target_z = dds.read('Z_dest')
        Robot_index = dds.read('Robot_index')
        Trajectory_Type = dds.read('Trajectory_Type')

        if None in (x, z, yaw, current_speed, target_x, target_z, Robot_index, Trajectory_Type):
            print('error')
            continue

        yaw_deg = math.degrees(yaw)
        dist_to_target = math.hypot(target_x - x, target_z - z)
        target_angle = math.atan2(target_x - x, target_z - z)
        target_angle = math.degrees(target_angle)
        angle_error = target_angle - yaw_deg

        # Speed adjustment based on distance and trajectory segment
        if dist_to_target < slow_radius:
            if Trajectory_Type == 1:
                target_speed = max(2.3, target_speed - 0.05)
            elif Trajectory_Type == 2:
                target_speed = max(0.5, target_speed - 0.05)
        else:
            target_speed = min(max_speed, target_speed + 0.03)

        # Waypoint arrival check
        if dist_to_target < acceptance_radius:
            print(f'Waypoint {Robot_index + 1} reached! ({target_x}, {target_z})')
            
            # If this was the last point on the path, stop the robot and exit
            if Trajectory_Type == 2:
                print('Final destination reached! Stopping robot and terminating.')
                dds.publish('Torque', 0, DDS.DDS_TYPE_FLOAT)
                dds.publish('Theta', 0, DDS.DDS_TYPE_FLOAT)
                break

            # Intermediate waypoint: advance index
            Robot_index += 1
            dds.publish('Control_Index', Robot_index, DDS.DDS_TYPE_INT)
            
        angle_error = (angle_error + 180) % 360 - 180
        raw_steering = steering_pid.evaluate(angle_error, delta_t)
        target_steering = max(-max_steering, min(max_steering, raw_steering))

        speed_error = target_speed - current_speed
        torque = speed_pid.evaluate(speed_error, delta_t, max_torque)

        # Send control commands to Godot
        dds.publish('Torque', torque, DDS.DDS_TYPE_FLOAT)
        dds.publish('Theta', target_steering, DDS.DDS_TYPE_FLOAT)

        d_time.append(t_elapsed)
        d_x.append(x)
        d_z.append(z)
        d_target_speed.append(target_speed)
        d_current_speed.append(current_speed)
        d_angle_error.append(angle_error)
        d_steering.append(target_steering)

except KeyboardInterrupt:
    print('interrotto')

Starting robot conrtrol...
error
Waypoint 1 reached! (0.0, 0.0)
Waypoint 1 reached! (0.0, 0.0)
Waypoint 2 reached! (10.0, 0.0)
Waypoint 2 reached! (10.0, 0.0)
Waypoint 3 reached! (20.0, 0.0)
Waypoint 3 reached! (20.0, 0.0)
Waypoint 4 reached! (20.0, 10.0)
Waypoint 4 reached! (20.0, 10.0)
Waypoint 5 reached! (20.0, 20.0)
Final destination reached! Stopping robot and terminating.


In [10]:
dds = DDS()
dds.start('127.0.0.1', 4444)
dds.subscribe(['tick', 'X', 'Z', 'Yaw', 'Speed', 'X_dest', 'Z_dest', 'Robot_index', 'Trajectory_Type'])

acceptance_radius = 2.0
final_acceptance_radius = 0.2

slow_radius = 5.0
max_speed = 3.0
target_speed = 3.0
max_torque = 150
max_deceleration = 1.0

speed_pid = PID_Controller(20.0, 100.0, 0.05)
steering_pid = PID_Controller(kp=1.0, ki=0.0, kd=0.0)

max_steering = 35

d_x, d_z, d_time = [], [], []
d_target_speed, d_current_speed = [], []
d_angle_error, d_steering = [], []

print("Starting robot control...")
start_time = time.time()

try:
    while True:
        delta_t = dds.wait('tick')
        if delta_t is None:
            continue
            
        t_elapsed = time.time() - start_time

        x = dds.read('X')
        z = dds.read('Z')
        yaw = dds.read('Yaw') 
        current_speed = dds.read('Speed')
        target_x = dds.read('X_dest')
        target_z = dds.read('Z_dest')
        Robot_index = dds.read('Robot_index')
        Trajectory_Type = dds.read('Trajectory_Type')

        if None in (x, z, yaw, current_speed, target_x, target_z, Robot_index, Trajectory_Type):
            print('error')
            continue

        yaw_deg = math.degrees(yaw)
        dist_to_target = math.hypot(target_x - x, target_z - z)
        target_angle = math.degrees(math.atan2(target_x - x, target_z - z))
        angle_error = target_angle - yaw_deg

        if Trajectory_Type == 2:
            decel_speed = math.sqrt(2 * max_deceleration * dist_to_target)
            target_speed = min(max_speed, decel_speed)
        elif dist_to_target < slow_radius:
            if Trajectory_Type == 1: # Curva, deceleriamo...
                target_speed = max(2.3, target_speed - 0.05)
        else:
            target_speed = min(max_speed, target_speed + 0.03)

        if Trajectory_Type == 2: # Stiamo approcciando l'ultimo punto.
            if dist_to_target <= final_acceptance_radius:
                print(f'Final destination reached!')
                dds.publish('Torque', 0, DDS.DDS_TYPE_FLOAT)
                dds.publish('Theta', 0, DDS.DDS_TYPE_FLOAT)
                break
        else:
            if dist_to_target < acceptance_radius:
                print(f'Waypoint {Robot_index + 1} reached! ({target_x}, {target_z})')
                Robot_index += 1
                dds.publish('Control_Index', Robot_index, DDS.DDS_TYPE_INT)

        angle_error = (angle_error + 180) % 360 - 180
        raw_steering = steering_pid.evaluate(angle_error, delta_t)
        target_steering = max(-max_steering, min(max_steering, raw_steering))

        speed_error = target_speed - current_speed
        torque = speed_pid.evaluate(speed_error, delta_t, max_torque)

        # Publish to Godot
        dds.publish('Torque', torque, DDS.DDS_TYPE_FLOAT)
        dds.publish('Theta', target_steering, DDS.DDS_TYPE_FLOAT)

        d_time.append(t_elapsed)
        d_x.append(x)
        d_z.append(z)
        d_target_speed.append(target_speed)
        d_current_speed.append(current_speed)
        d_angle_error.append(angle_error)
        d_steering.append(target_steering)

except KeyboardInterrupt:
    print('Interrupted')

Starting robot control...
error
Final destination reached! Distance error: 0.17 units.
